<a href="https://colab.research.google.com/github/dymiyata/wva-ml-ai-1-2026-2027/blob/main/logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression

Here we learn how to perform logistic regression using sklearn

In [1]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Now we import the dataset we will use.  This is a dataset built in to sklearn.  It has data on a bunch of tumors.  

- We will load all the features into a DataFrame called `X`.  
- The target variable is whether or not the tumor is malignant (cancerous)
  - We have avalue of `1` for malignant
  - We have a value of `0` for not malignant (aka benign).
- We'll load the values of the target variable into `y`

In [3]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

In [6]:
y.tail()

,0
564,0
565,0
566,0
567,0
568,1


### Split into Training and Testing sets

Now we must split the data into a training set and a testing set.  Let's do a 75/25 train test split.  We'll use 2026 for the `random_state`.

- We'll need to add one more argument to `train_test_split` which is `stratify=y`.  What do you think this does?

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=2026,
    test_size=0.25,
    stratify=y
)

`stratify=y` just ensures the proportion of 0s and 1s is about the same in the training and testing sets.

### Preprocessing the Data

Recall, in linear regression it's important for your features to be similar in scale. Why?

The same holds true for logistic regression.  Before I had you just manually and imprecisely scale the features, there is a better way to do things.



To do this, we will take each feature variable $x$ then:
- subtract its mean $\bar{x}$ from each training example so that the mean gets shifted to $0$
- divide each training example by the standard deviation $s$ so that the standard deviation becomes 1.

In this way, you will replace each feature $x$ with $$\frac{x - \bar{x}}{s}.$$  Thus, each resulting feature will have mean 0 and standard deviation 1 so they will all be on the same scale.

Luckily, `sklearn` has a built-in way to do this automatically:

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
scaler = StandardScaler()

scaler.fit(X_train) # scaler is learning mean and st. dev. for the train set

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [10]:
X_train[:10]

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
288,11.260,19.96,73.72,394.1,0.08020,0.11810,0.092740,0.05588,0.2595,0.06233,...,11.86,22.33,78.27,437.6,0.10280,0.1843,0.15460,0.09314,0.2955,0.07009
424,9.742,19.12,61.93,289.7,0.10750,0.08333,0.008934,0.01967,0.2538,0.07029,...,11.21,23.17,71.79,380.9,0.13980,0.1352,0.02085,0.04589,0.3196,0.08009
208,13.110,22.54,87.02,529.4,0.10020,0.14830,0.087050,0.05102,0.1850,0.07310,...,14.55,29.16,99.48,639.3,0.13490,0.4402,0.31620,0.11260,0.4128,0.10760
171,13.430,19.63,85.84,565.4,0.09048,0.06288,0.058580,0.03438,0.1598,0.05671,...,17.98,29.87,116.60,993.6,0.14010,0.1546,0.26440,0.11600,0.2884,0.07371
235,14.030,21.25,89.79,603.4,0.09070,0.06945,0.014620,0.01896,0.1517,0.05835,...,15.33,30.28,98.27,715.5,0.12870,0.1513,0.06231,0.07963,0.2226,0.07617
353,15.080,25.74,98.00,716.6,0.10240,0.09769,0.123500,0.06553,0.1647,0.06464,...,18.51,33.22,121.20,1050.0,0.16600,0.2356,0.40290,0.15260,0.2654,0.09438
168,17.470,24.68,116.10,984.6,0.10490,0.16030,0.215900,0.10430,0.1538,0.06365,...,23.14,32.33,155.30,1660.0,0.13760,0.3830,0.48900,0.17210,0.2160,0.09300
537,11.690,24.44,76.37,406.4,0.12360,0.15520,0.045150,0.04531,0.2131,0.07405,...,12.98,32.19,86.12,487.7,0.17680,0.3251,0.13950,0.13080,0.2803,0.09970
93,13.450,18.30,86.60,555.1,0.10220,0.08165,0.039740,0.02780,0.1638,0.05710,...,15.10,25.94,97.59,699.4,0.13390,0.1751,0.13810,0.07911,0.2678,0.06603
290,14.410,19.73,96.03,651.0,0.08757,0.16760,0.136200,0.06602,0.1714,0.07192,...,15.77,22.13,101.70,767.3,0.09983,0.2472,0.22200,0.10210,0.2272,0.08799


In [11]:
X_train_scaled[:10]

array([[-0.79043824,  0.13774499, -0.72878826, -0.7231638 , -1.1677208 ,
         0.29106611,  0.06024964,  0.18533245,  3.06696789, -0.03063775,
         0.25484057,  1.24751238, -0.01011719, -0.1389047 ,  3.06857784,
         3.28374615,  1.89662368,  4.00400972,  1.83234643,  1.13824321,
        -0.88961345, -0.55577803, -0.84182532, -0.76222049, -1.32086404,
        -0.42832305, -0.5452161 , -0.31250645,  0.13296378, -0.76361944],
       [-1.21147418, -0.05213392, -1.20310801, -1.01154462,  0.8530337 ,
        -0.37736034, -0.99657155, -0.74221615,  2.84727707,  1.18917439,
         0.96476363,  0.95906556,  0.79935025,  0.03917687,  2.12668025,
        -0.35177088, -0.93858261, -0.17883094,  1.41705783,  0.2036197 ,
        -1.02009552, -0.42208554, -1.02955789, -0.85798318,  0.34384168,
        -0.73622168, -1.17904478, -1.02623386,  0.55098372, -0.20595589],
       [-0.27731801,  0.72094453, -0.19372018, -0.34942889,  0.31268543,
         0.87163783, -0.01150313,  0.0608396 ,  0

### Training the Logistic Regression model

Now that we've split the data into training and testing sets and preprocessed the data, we're ready to train the model.  

This part will be almost the same as it was for linear regression:

First we import and define the logistic regression model:

In [12]:
from sklearn.linear_model import LogisticRegression

Now we define the model and fit it to the data.

In [18]:
model = LogisticRegression(max_iter=10000)

In [23]:
model.fit(X_train_scaled, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(max_iter=10000)

With logistic regression, our model predicts probabilities that $y=1$ meaning probabilities that tumors are malignant.  

We can access these values as follows:

In [24]:
p_train = model.predict_proba(X_train_scaled)
p_test = model.predict_proba(X_test_scaled)

In [25]:
p_train[:10]

array([[3.27637795e-04, 9.99672362e-01],
       [1.24888113e-03, 9.98751119e-01],
       [2.16344352e-01, 7.83655648e-01],
       [9.12466976e-01, 8.75330237e-02],
       [2.47733521e-02, 9.75226648e-01],
       [9.99499307e-01, 5.00693264e-04],
       [9.99999518e-01, 4.81702473e-07],
       [8.21160051e-02, 9.17883995e-01],
       [3.19603200e-02, 9.68039680e-01],
       [1.42036959e-04, 9.99857963e-01]])

To get the actual predictions (just 0 or 1), we compare the probabilities to a threshold value (by defualt we use a threshould of 0.5).
- values above the threshold are predicted to be 1.
-values below the threshould are predicted to be 0.

`sklearn` does this automatically with just `model.predict()`

In [26]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

In [27]:
y_pred_train[:10]

array([1, 1, 1, 0, 1, 0, 0, 1, 1, 1])

### Evaluating the model

Now how do we evaluate the model?

A good way is to check our accuracy on the test set.
- Accuracy just means what percentage of examples we classify correctly.

`sklearn` has a nice function to help us compute this called `accuracy_score`.  Let's import it and use it:

In [31]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [29]:
accuracy_score(y_train, y_pred_train)

0.9906103286384976

In [30]:
accuracy_score(y_test, y_pred_test)

0.972027972027972

Another nice visualization is something called the "Confusion Matrix".  Let's see how that works

In [32]:
confusion_matrix(y_train, y_pred_train)

array([[155,   4],
       [  0, 267]])

The confusion matrix outputs a $2\times 2$ matrix showing True/False Positives/Negatives.  The order goes:
$$\begin{bmatrix}
\text{TN} & \text{FP} \\
\text{FN} & \text{TP}
\end{bmatrix}$$

I always forget the order and have to look it up...